# Boreal vegetation height: v3.1 maps + VHR
+ v3.1 circumpolar vegetation height maps (30 m) for every available year
+ Vegetation height from VHR (2 m), imported to the MAAP user STAC with the `GenerateCogStacItems` DPS algorithm (see `DPS_import_COGs_to_stac.ipynb`)

Both use the same binned height colormap. Bin edges are defined in **meters**:
+ v3.1 height maps are in meters → bins used as-is
+ VHR rasters are in **decimeters** → bin edges ×10 for the tiler

The colorbar spans 0-15 m.

In [ ]:
!pip install cogeo_mosaic==8.2.0

In [ ]:
import os
import sys
import json
import urllib
from datetime import datetime

import geopandas
import httpx
import numpy as np
import matplotlib.colors as mcolors
from folium import TileLayer

ICESAT2_BOREAL_LIB_PATH = '/projects/my-private-bucket/code/icesat2_boreal/lib'
sys.path.append(ICESAT2_BOREAL_LIB_PATH)
import maplib_folium
import mosaiclib

## Height colormap

In [ ]:
CBAR_MAX_M = 15

# Bin edges [m] and colors (gray -> orange -> yellow -> greens)
HT_BINS_M = [(0.0, 3.5), (3.5, 10), (10, 15), (15, 30), (30, 40), (40, 50), (50, 600)]
HT_COLORS = [
    (99, 99, 99),      # Gray
    (253, 153, 96),    # Light orange
    (253, 234, 156),   # Light yellow
    (235, 246, 164),   # Yellow-green
    (167, 217, 109),   # Light green
    (43, 161, 82),     # Medium green
    (0, 90, 50)        # Dark green
]

def make_binned_colormap(bins, colors, sub_bins_per_class=6, scale=1):
    '''
    Binned colormap with a gradient within each bin toward the next bin's color.
    scale multiplies bin edges (e.g., 10 for data stored in decimeters).
    Returns a titiler interval colormap: [[[min, max], [r, g, b, a]], ...]
    '''
    cmap = []
    for i, (min_h, max_h) in enumerate(bins):
        color_curr = colors[i]
        color_next = colors[i + 1] if i < len(bins) - 1 else color_curr
        edges = np.linspace(min_h * scale, max_h * scale, sub_bins_per_class + 1)
        for j in range(sub_bins_per_class):
            f = j / sub_bins_per_class
            rgb = [int(c0 + (c1 - c0) * f) for c0, c1 in zip(color_curr, color_next)]
            cmap.append([[round(edges[j], 3), round(edges[j + 1], 3)], rgb + [255]])
    return cmap

def interval_cmap_to_mpl(interval_cmap, vmin, vmax, n=256):
    '''Sample an interval colormap into a matplotlib colormap so the folium colorbar matches the tiles.'''
    rgb = []
    for v in np.linspace(vmin, vmax, n):
        color = next((c for (b0, b1), c in interval_cmap if b0 <= v < b1), interval_cmap[-1][1])
        rgb.append([x / 255 for x in color[:3]])
    return mcolors.ListedColormap(rgb)

HT_CMAP_M  = make_binned_colormap(HT_BINS_M, HT_COLORS)            # v3.1 maps [m]
HT_CMAP_DM = make_binned_colormap(HT_BINS_M, HT_COLORS, scale=10)  # VHR [dm]
HT_CMAP_MPL = interval_cmap_to_mpl(HT_CMAP_M, 0, CBAR_MAX_M)

## v3.1 vegetation height layers
One layer per year found in `mosaiclib.HT_MOSAIC_JSON_FN_DICT`. Note: zoom-in to display map tiles.

In [ ]:
HT_YEARS = sorted(int(k.split('_')[0]) for k in mosaiclib.HT_MOSAIC_JSON_FN_DICT if k.endswith('_v3.1_multiyr'))
print(HT_YEARS)

HT_TILES_LAYER_LIST = []
HT_POINT_URL_DICT = {}
for i, YEAR in enumerate(HT_YEARS):
    NAME = f"Vegetation height {YEAR} [m] v3.1"
    tiles = maplib_folium.build_tiles_with_params(
        None,
        mosaiclib.HT_MOSAIC_JSON_FN_DICT[f'{YEAR}_v3.1_multiyr'],
        params_dict={"bidx": "1", "colormap": json.dumps(HT_CMAP_M)},
        DEBUG=False,
    )
    HT_POINT_URL_DICT[YEAR] = tiles.split('/tiles/')[0] + '/point/{lon},{lat}?bidx=1'
    HT_TILES_LAYER_LIST.append({
        'layer': TileLayer(tiles=tiles, name=NAME, attr="MAAP", overlay=True, opacity=1, show=False),
        'cmap': HT_CMAP_MPL,
        'min_val': 0,
        'max_val': CBAR_MAX_M,
        'caption': "Vegetation height [m] (v3.1 maps and VHR)",
        'show_cbar': i == 0,   # one shared colorbar for all height layers
    })

## Vegetation height from VHR layer
Tiles come from the titiler-pgstac endpoint for DPS user collections. Request the collection `tilejson.json` to get the XYZ tile template (not the `map.html` viewer page).

In [ ]:
DPS_USER_STAC_TITILER_ENDPOINT = "https://ansdoe39vf.execute-api.us-west-2.amazonaws.com"

VHR_IDENTIFIER = '002m_mos_2018_212_deltayr8_months78_02m'
VHR_COLLECTION_ID = f'montesano__GenerateCogStacItems__v0.1__{VHR_IDENTIFIER}'.lower().replace('/','_')
VHR_ASSET_NAME = "asset"
VHR_NAME = "Vegetation height from VHR [m]"
VHR_BBOX = [-170, 55, -140, 70]   # min_lon, min_lat, max_lon, max_lat
VHR_MIN_ZOOM = 8                  # 2 m data: don't request tiles at continental zooms

VHR_RENDER_PARAMS = {
    "assets": f"{VHR_ASSET_NAME}|bidx=1",
    "colormap": json.dumps(HT_CMAP_DM),   # decimeter bin edges
}

r = httpx.get(
    f"{DPS_USER_STAC_TITILER_ENDPOINT}/collections/{VHR_COLLECTION_ID}/WebMercatorQuad/tilejson.json",
    params={**VHR_RENDER_PARAMS, "minzoom": VHR_MIN_ZOOM},
    timeout=60,
)
if r.status_code == 200:
    vhr_tiles = r.json()['tiles'][0]
else:
    print(f'tilejson request failed ({r.status_code}): {r.text[:300]}')
    vhr_tiles = (f"{DPS_USER_STAC_TITILER_ENDPOINT}/collections/{VHR_COLLECTION_ID}"
                 "/tiles/WebMercatorQuad/{z}/{x}/{y}?" + urllib.parse.urlencode(VHR_RENDER_PARAMS))

min_lon, min_lat, max_lon, max_lat = VHR_BBOX
VHR_TILES_LAYER_LIST = [{
    'layer': TileLayer(
        tiles=vhr_tiles, name=VHR_NAME, attr="NASA MAAP", overlay=True, opacity=1, show=False,
        min_zoom=VHR_MIN_ZOOM,
        bounds=[[min_lat, min_lon], [max_lat, max_lon]],
    ),
    'cmap': HT_CMAP_MPL,
    'min_val': 0,
    'max_val': CBAR_MAX_M,
    'caption': VHR_NAME,
    'show_cbar': False,   # shares the v3.1 height colorbar
}]

# Map

In [ ]:
boreal_tiles = geopandas.read_file(mosaiclib.boreal_tile_index_path)

m = maplib_folium.MAP_REGISTERED_DPS_RESULTS(
                                boreal_tiles,
                                None,
                                CHECK_TILES_NAME='',
                                map_width=750, map_height=500, SHOW_WIDGETS=False,
                                ADD_TILELAYER = HT_TILES_LAYER_LIST + VHR_TILES_LAYER_LIST   # VHR last = on top
                             )

## Pixel query

In [ ]:
# VHR values are returned in raw decimeters
query_layers = [
    {'section': 'Vegetation height from VHR', 'name': 'height',
     'point_url': f"{DPS_USER_STAC_TITILER_ENDPOINT}/collections/{VHR_COLLECTION_ID}/point/{{lon}},{{lat}}?assets={VHR_ASSET_NAME}",
     'units': 'dm'},
] + [
    {'section': 'Vegetation height v3.1', 'name': str(YEAR), 'point_url': HT_POINT_URL_DICT[YEAR], 'units': 'm'}
    for YEAR in HT_YEARS
]

m = maplib_folium.add_pixel_query(m, query_layers, decimals=1)
m

### Save to html

In [ ]:
out_html = '/projects/my-private-bucket/code/maap_tools/maps/Review_maps_v3.1_height_VHR.html'

m.save(out_html)

header_html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
            max-width: 1600px; margin: 0 auto; padding: 24px 32px;">
  <h1 style="margin-bottom: 8px;">Boreal vegetation height</h1>
  <h3 style="margin-top: 0; color: #555; font-weight: normal;">NASA ABoVE Boreal Biomass Mapping Project</h3>
  <div style="font-size: 14px; line-height: 1.5; color: #333;">
    <ul>
      <li><b>Vegetation height, {HT_YEARS[0]}–{HT_YEARS[-1]} (v3.1)</b> — annual 30 m vegetation height (m) from ICESat-2 ~ HLS modelling.</li>
      <li><b>Vegetation height from VHR</b> — 2 m vegetation height from very-high-resolution imagery. Zoom in (level {VHR_MIN_ZOOM}+) to display.</li>
    </ul>
    <p>All height layers share one colormap and colorbar (0–{CBAR_MAX_M} m). Use the layer-control panel (top right) to toggle layers; click the map to query pixel values.</p>
  </div>
  <div style="display: flex; justify-content: space-between; align-items: center;
              margin-top: 18px; padding-top: 12px; border-top: 1px solid #e0e6ee;
              font-size: 0.85em; color: #718096;">
    <span style="font-style: italic;">Work in progress</span>
    <span><b>Contact:</b> <a href="mailto:paul.m.montesano@nasa.gov">paul.m.montesano@nasa.gov</a></span>
    <span><b>Last updated:</b> {datetime.now().strftime('%Y-%m-%d %H:%M UTC')}</span>
  </div>
</div>
"""

fullwidth_css = """
<style>
  body { margin: 0; padding: 0; }
  .folium-map, div[id^="map_"] {
    width: 100% !important;
    height: 60vh !important;
    max-width: 100% !important;
  }
</style>
"""

with open(out_html, 'r') as f:
    html = f.read()
html = html.replace('<body>', f'<body>\n{fullwidth_css}\n{header_html}', 1)
with open(out_html, 'w') as f:
    f.write(html)

print(f"Saved {out_html} — push to GitHub to view at:")
print(f"  https://pahbs.github.io/maap_tools/maps/{os.path.basename(out_html)}")